# llms

> Generic LLM-calling utilities (models, prompting, tool schemas) -- deliberately kept independent of boopiter's own Notebook/Cell model, so this module never imports from `cells.py`. `cells.py` imports from here, never the other way around.

In [ ]:
#| default_exp llms

In [ ]:
#| export
import inspect, time
from fastcore.utils import *
from lisette import *

In [ ]:
#| export
# slmn (github.com/drscotthawley/slmn, a separate general-purpose toolkit -- see
# boopiter/pyproject.toml for the git dependency) is imported as plain modules here, not
# re-exported via `import *` -- see get_tool_list() below, which assembles actual tool
# functions on demand instead of merging namespaces.
import slmn.nbtools as _slmn_nbtools
import slmn.misc as _slmn_misc
import slmn.remote as _slmn_remote
import slmn.dead_drop as _dd

# slmn.remote also has remote_launch/remote_status/remote_smoke_test, which can run arbitrary
# commands on a remote host over ssh -- a much bigger capability than the rest of slmn's tools.
# Least-privilege: don't hand those to the LLM by default, just the read-only/informational ones.
_SLMN_REMOTE_SAFE = ('fetch_url', 'check_ci', 'remote_gpu_free')

_LOCAL_TOOLS = []  # boopiter-defined tools always available, independent of slmn; empty for now, add as needed

In [ ]:
#| export
DEFAULT_TOOL_SELECTION = {'boopiter': True, 'slmn-nbtools': True, 'slmn-misc': True, 'slmn-remote': False}  # matches the wrench-icon Tools menu's default checkbox states

def get_tool_list(selection:dict=None # which tool sources to include -- keys 'boopiter'/'slmn-nbtools'/'slmn-misc'/'slmn-remote', bool values (see the wrench-icon Tools menu in the GUI). Defaults to DEFAULT_TOOL_SELECTION if omitted. 'slmn-remote', even when selected, only ever contributes its safe/read-only subset (_SLMN_REMOTE_SAFE) -- remote_launch/remote_status/remote_smoke_test (arbitrary remote command execution over ssh) are never included here, by design, regardless of selection.
                   ) -> list:
    "Assemble the list of tool functions to offer an LLM, from whichever sources are selected. Returns actual callables, not names -- pass straight to prompt_llm(tools=...). Per-notebook ad-hoc tools (see add_tool()) are layered on top of this by the caller, not included here."
    selection = selection if selection is not None else DEFAULT_TOOL_SELECTION
    tools = []
    if selection.get('boopiter'): tools += _LOCAL_TOOLS
    if selection.get('slmn-nbtools'): tools += [getattr(_slmn_nbtools, name) for name in _slmn_nbtools.__all__]
    if selection.get('slmn-misc'): tools += [getattr(_slmn_misc, name) for name in _slmn_misc.__all__]
    if selection.get('slmn-remote'): tools += [getattr(_slmn_remote, name) for name in _SLMN_REMOTE_SAFE]
    return tools

In [ ]:
#| export
def get_ollama_list() -> list[dict]:
    "Get info on every locally-available Ollama model, straight from /api/tags -- each dict is the raw entry (name, details incl. parameter_size/family, and capabilities e.g. 'vision'/'tools'/'thinking') plus an added 'id' key ('ollama/<model>', the string used elsewhere as the model identifier -- see nb.model). Returns [] and warns if Ollama unavailable."
    import httpx, warnings
    try:
        models = httpx.get("http://localhost:11434/api/tags").json().get('models', [])
        for m in models: m['id'] = 'ollama/' + m['model']
        return models
    except Exception as e:
        warnings.warn(f"Ollama not available: {e}")
        return []

In [ ]:
#| export
def get_model_list() -> list[dict]:
    "Wrapper routine to get info dicts (see get_ollama_list()) for all available models from all sources."
    deaddrop = [{'id': 'deaddrop/claude', 'model': 'claude', 'capabilities': ['vision', 'thinking']}]
    return get_ollama_list() + deaddrop  # TODO: add more model source, e.g. cloud, fileio

In [ ]:
#| eval: false
get_model_list()

['ollama/qwen2.5-coder:latest',
 'ollama/gemma3:4b',
 'ollama/llama3.1:latest',
 'ollama/qwen2.5:latest']

In [ ]:
#| export
# lisette + local (Ollama) models: passing non-empty `tools=` combined with `tool_choice='none'`
# triggers a bug in litellm's MCP-handler codepath that returns a raw dict instead of a proper
# response object (AttributeError: 'dict' object has no attribute 'choices'). Same issue hit by
# SBrewer15/CellMate (https://github.com/SBrewer15/CellMate) -- this is their patch, adopted as-is:
# drop tool_schemas for just that one call whenever tool_choice=='none', then restore them after.
_orig_chat_call = Chat._call


In [ ]:
#| export
@patch
def _call(self:Chat, msg:str|None=None, prefill:str|None=None, temp:float|None=None, think:str|None=None,
          search:str|None=None, stream:bool=False, max_steps:int=2, step:int=1, final_prompt:dict|None=None,
          tool_choice:str|None=None, max_tokens:int|None=None, **kwargs):
    "Internal method that always yields responses -- patched (see the comment above) to avoid a litellm/Ollama tool-calling bug."
    _orig_tools = self.tool_schemas
    if tool_choice == 'none': self.tool_schemas, tool_choice = None, None
    try: yield from _orig_chat_call(self, msg, prefill, temp, think, search, stream, max_steps, step, final_prompt, tool_choice, max_tokens, **kwargs)
    finally: self.tool_schemas = _orig_tools

In [ ]:
#| export
def _reply_details_html(response, msg) -> str:
    "A collapsible <details> block summarizing an LLM call's metadata (model, finish reason, token counts, tool calls, reasoning) -- not part of the reply's actual content, kept out of Cell.source (see Cell.details) so it's never sent back to the model as context, and shown collapsed, in gray, above the real text. onclick=stopPropagation keeps a click on <summary> from also bubbling into the cell's click-anywhere-to-edit handler."
    u = response.usage
    rows = [('Model', response.model), ('Finish reason', response.choices[0].finish_reason)]
    if u: rows.append(('Tokens', f'{u.prompt_tokens} prompt + {u.completion_tokens} completion = {u.total_tokens} total'))
    if msg.tool_calls: rows.append(('Tool calls', ', '.join(tc.function.name for tc in msg.tool_calls)))
    items = ''.join(f'<li>{k}: {v}</li>' for k, v in rows)
    reasoning = f'<pre style="white-space:pre-wrap">{msg.reasoning_content}</pre>' if getattr(msg, 'reasoning_content', None) else ''
    return (f'<details class="text-gray-400" onclick="event.stopPropagation()"><summary>Reply details</summary>'
            f'<ul>{items}</ul>{reasoning}</details>')

### Why tools aren't passed as native API `tools=`

Early on, enabling any tool source made local models (tested worst-case: `qwen2.5-coder:latest`, but reproduced up through `qwen3.6:27b`) reflexively try to call a tool on *every* prompt, including plain "write me some code" requests that had nothing to do with any tool -- producing garbled, empty, or JSON-as-prose replies instead of a normal answer.

The root cause: Ollama/Qwen's chat template wraps any non-empty `tools=` API parameter into a `<tools>...</tools>` special-token block. That structural block biases these models into feeling obligated to fill a `<tool_call>` slot, regardless of `tool_choice` (`'auto'` vs. the default `None` made no measurable difference) and regardless of a system prompt saying "only use a tool if you actually need one" (that improved *presentation* -- coherent prose vs. raw JSON -- but didn't stop the reflex). Bigger models weren't immune either, just failed differently (e.g. the tool call landing in `reasoning_content` while `content` came back empty).

The fix, found by reading how SolveIt's `dialoghelper`/`pyskills` solve the same problem: never populate the native `tools=` parameter. Instead, describe the available functions in the system prompt as plain Python callables already present in the execution namespace (see `tools_system_prompt()` below), and let the model express "I need a tool" by writing an ordinary ```python code block as part of its reply -- the same code-generation skill these models are already reliably good at, rather than their much less reliable native function-calling judgment. `stream_llm_reply()` (and `_push_tools()` in `cells.py`, which keeps those functions live in the shared kernel namespace) implement that pattern end to end.

In [ ]:
#| export
def _tool_doc_line(fn) -> str:
    "One-line description of a tool function for the system prompt: name, signature, and the first line of its docstring."
    sig = str(inspect.signature(fn))
    doc = (inspect.getdoc(fn) or '').split('\n')[0]
    return f"- {fn.__name__}{sig}: {doc}"

def tools_system_prompt(tools:list|None) -> str:
    "Build a system-prompt block describing `tools` as plain Python functions already available in the model's execution environment -- NOT passed as native API tool schemas. This sidesteps a real reliability problem found via testing: Ollama/Qwen's chat template wraps any non-empty `tools=` into a `<tools>` special-token block that biases even capable local models into reflexive, often-wrong tool calls, regardless of tool_choice or how carefully the system prompt says 'only if needed' (both were tried and didn't help). Routing tool use through the model's own code-writing judgment instead -- which local models are much more reliably good at -- fixed it in practice. Same approach dialoghelper/pyskills use for SolveIt (not a dependency here, just the inspiration). Returns '' if `tools` is empty, so no system prompt is added at all when there's nothing to describe."
    if not tools: return ''
    lines = '\n'.join(_tool_doc_line(fn) for fn in tools)
    return (
        "The following Python functions are already available in your code execution environment "
        "(no import needed) if a task genuinely requires one:\n"
        f"{lines}\n\n"
        "To use one, write ordinary Python code calling it inside a fenced ```python code block, "
        "as part of your normal reply -- the user can run that code directly. Do not describe a "
        "tool call as JSON or any other structured format; just write real Python code, and only "
        "when a task actually needs it. For requests that don't require reading/editing files or "
        "other tool-backed actions -- e.g. 'write me some code', general questions -- just answer "
        "directly; these helper functions are irrelevant to those."
    )

def prompt_llm(context:str, model:str='ollama/qwen2.5-coder:latest', tools:list|None=None, think:str|None=None) -> tuple[str,str]:
    "Send a prompt to the LLM; returns (content, details_html) -- the reply text itself, and a separate collapsible <details> block of call metadata (model/tokens/finish reason/reasoning) meant to be stored apart from the reply (see Cell.details), not mixed into it. `tools`, if given, are described via tools_system_prompt() (code-callable, not native tool-calling) -- see there for why. `think`, if given ('l'/'m'/'h'), is lisette's own reasoning-effort control -- passed straight through to litellm, which maps it to Ollama's 'think' request field; only meaningful for a model that actually supports thinking (see the brain-icon reasoning-model picker)."
    # _skip_mcp_handler avoids litellm's MCP-proxy import chain (needs fastapi/orjson) that we don't use.
    # Drop it (and re-add fastapi/orjson to pyproject.toml) if/when we actually want MCP tool support.
    chat = Chat(model, sp=tools_system_prompt(tools), tools=[], callkw={'_skip_mcp_handler': True}) # FYI: this makes a fresh stateless context each time. is that what we want?
    response = chat(context, think=think)
    msg = contents(response)
    return msg.content, _reply_details_html(response, msg)

_DEADDROP_DIR = Path.home() / 'dead_drop'  # prompts/ and responses/ subdirs -- see slmn.dead_drop

def _deaddrop_stream_reply(context:str, model:str, poll_interval:float=1.0):
    "Route a Prompt-cell reply through the dead_drop protocol (github.com/drscotthawley/slmn) instead of a local LLM call: drop `context` as a prompt (slmn.dead_drop.drop(), auto-named), then poll the paired response file (same name, under _DEADDROP_DIR/responses) for new content, yielding ('delta', ...) chunks as it grows -- same idea as run_code_poll's streaming, but the 'model' on the other end is a human relaying a real Claude session through files. The responder signals completion with a trailing '---DONE---' line (same sentinel convention as slmn.dead_drop.next_prompt()'s '---SEND---'); everything before it becomes the final reply. There's no API response object to summarize, so details_html is just a minimal <details> block naming `model` -- kept in the same collapsible spot as `_reply_details_html`'s real metadata, for header uniformity with the Ollama path (see cell_header() in cells.py)."
    path = _dd.drop(str(_DEADDROP_DIR / 'prompts'), context)
    resp_path = _DEADDROP_DIR / 'responses' / Path(path).name
    details_html = (f'<details class="text-gray-400" onclick="event.stopPropagation()"><summary>Reply details</summary>'
                     f'<ul><li>Model: {model}</li></ul></details>')
    seen = 0
    while True:
        if resp_path.exists():
            text = resp_path.read_text()
            stripped = text.rstrip()
            if stripped.endswith('---DONE---'):
                content = stripped[:-len('---DONE---')].rstrip()
                if len(content) > seen: yield ('delta', content[seen:])
                yield ('final', content, details_html)
                return
            if len(text) > seen:
                yield ('delta', text[seen:])
                seen = len(text)
        time.sleep(poll_interval)

def stream_llm_reply(context:str, model:str, tools:list|None=None, think:str|None=None):
    "Generator streaming a model reply token-by-token, using the code-callable tool strategy (see tools_system_prompt) instead of native tool-calling. Yields ('delta', text) chunks as they arrive, then a final ('final', content, details_html) tuple once the response completes. Callers (e.g. cells.py's _run_prompt_bg) drive this into their own background-thread/UI state -- that part isn't LLM-specific, so it stays out of this module. `think` ('l'/'m'/'h' or None) -- see prompt_llm(). A `model` starting with 'deaddrop/' is routed through _deaddrop_stream_reply() instead of a real API call -- everything else about this function's contract (the yielded tuple shapes) is identical either way, so no caller needs to know or care which one answered."
    if model.startswith('deaddrop/'):
        yield from _deaddrop_stream_reply(context, model)
        return
    chat = Chat(model, sp=tools_system_prompt(tools), tools=[], callkw={'_skip_mcp_handler': True})
    for chunk in chat(context, stream=True, think=think):
        if hasattr(chunk.choices[0], 'message'):  # the final item -- a full ModelResponse, not a delta
            msg = contents(chunk)
            yield ('final', msg.content, _reply_details_html(chunk, msg))
        else:
            delta = chunk.choices[0].delta.content
            if delta: yield ('delta', delta)

In [ ]:
#| export
_PREFERRED_MODEL_SUBSTR = 'qwen2.5-coder'  # used if present, regardless of exact tag/version


In [ ]:
#| eval: false
s ="Today is July 18. Who's one famous person with this birthday?"
c = prompt_llm(s) 
print(str(c[0]))
s = """
    Tell me the previous question I asked you, from the previous prompt. 
    I want to see if you retain state between calls"""
c = prompt_llm(s) 
print(str(c[0]))

One famous person born on July 18th is Mark Zuckerberg, the co-founder and CEO of Facebook.
I'm sorry for any confusion, but as an AI language model, I don't have the capability to remember or retain information across separate interactions. Each response is generated independently based on the input provided in each session. If you have a specific question or need assistance with something particular, feel free to ask!


### Tool Use

Example tool from lisette docs:

In [ ]:
def add_numbers(
    a: int,  # First number to add
    b: int   # Second number to add  
) -> int:
    "Add two numbers together"
    return a + b

In [ ]:
#| eval: false
res = prompt_llm("What's 47 + 23? Use the tool.", tools=[add_numbers])
print(res[0])

The sum of 47 and 23 is 70. I have completed my task as requested. If you need further assistance or have additional questions, feel free to ask!


In [ ]:
#| eval: false
# Confirms get_tool_list() assembles real, callable tools according to a selection dict.
from boopiter.llms import get_tool_list, DEFAULT_TOOL_SELECTION
print(len(get_tool_list()), "tools with defaults:", [t.__name__ for t in get_tool_list()])
print(len(get_tool_list({'boopiter': True, 'slmn-nbtools': False, 'slmn-misc': False, 'slmn-remote': True})), "tools with only boopiter+remote")